# Introduction to Retrieval Augmented Generation with S&P 500 news

In this notebook, you will explore how to build a simple Retrieval-Augmented Generation (RAG) pipeline using financial news articles from S&P 500 companies.

We'll start by vectorizing text data, creating a vector store using FAISS, and integrating it with OpenAI's GPT models to answer questions using retrieved information.

This workflow emulates real-world systems in finance where natural language data (news, filings, analyst reports) are used to support decision-making.

# 📌 Objectives

By the end of this notebook, students will be able to:

1. **Perform Semantic Search with Metadata Filtering:**
   - Query the provided FAISS vector store to retrieve relevant financial news articles based on natural language questions.
   - Apply optional filters using metadata such as ticker or publication date to refine search results.

2. **Enrich Data with Company Metadata:**
   - Use the `yfinance` library to retrieve company-level metadata (company name, sector, industry) for tickers in the dataset.
   - Integrate this metadata to support enhanced filtering and analysis of news data.

3. **Build a Retrieval-Augmented Generation (RAG) Pipeline:**
   - Combine retrieved news snippets as context to generate answers using OpenAI’s GPT models.
   - Construct effective prompts that guide the language model to provide concise, context-aware responses.

4. **Evaluate and Analyze RAG Outputs:**
   - Review generated answers alongside the supporting news excerpts.
   - Reflect on the strengths and limitations of the simple RAG pipeline and consider potential improvements, such as adding more filters or refining retrieval strategies.

5. **Incorporate Financial Metadata into Retrieval Context:**
   - Enrich retrieved news snippets with key financial metadata including ticker, company name, sector, and industry.
   - Format prompts that combine both text excerpts and metadata to provide richer context to the language model.

6. **Generate Context-Aware Answers Using OpenAI Models:**
   - Construct and send prompts to an LLM that leverage both news content and metadata to produce concise, informed financial analysis.

7. **Compare Answers With and Without Metadata:**
   - Evaluate the impact of including financial metadata on answer quality using criteria such as clarity, detail, accuracy, and contextual relevance.
   - Summarize findings to reflect on the role of metadata in improving retrieval-augmented generation.

## Install and Import important librairies

First, we install and import the necessary libraries for:
- Text embedding generation (sentence-transformers)
- Efficient similarity search (faiss)
- Data manipulation (pandas, numpy)
- Visualization (matplotlib)

> ℹ️ FAISS uses inner product for cosine similarity by normalizing vectors.

In [ ]:
%pip install sentence-transformers
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import matplotlib.pyplot as plt
import faiss

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load news data
We load a CSV file of financial news, focusing on TITLE and SUMMARY, along with metadata like TICKER and PUBLICATION_DATE.
These will be embedded into vectors and used for semantic retrieval.

In [ ]:
K = 25

In [ ]:
df_news = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Fintech/df_news.csv')
df_news['PUBLICATION_DATE'] = pd.to_datetime(df_news['PUBLICATION_DATE']).dt.date
display(df_news)

,TICKER,TITLE,SUMMARY,PUBLICATION_DATE,PROVIDER,URL
0,MMM,2 Dow Jones Stocks with Promising Prospects an...,The Dow Jones (^DJI) is made up of 30 of the m...,2025-05-29,StockStory,https://finance.yahoo.com/news/2-dow-jones-sto...
1,MMM,3 S&P 500 Stocks Skating on Thin Ice,The S&P 500 (^GSPC) is often seen as a benchma...,2025-05-27,StockStory,https://finance.yahoo.com/news/3-p-500-stocks-...
2,MMM,3M Rises 15.8% YTD: Should You Buy the Stock N...,"MMM is making strides in the aerospace, indust...",2025-05-22,Zacks,https://finance.yahoo.com/news/3m-rises-15-8-y...
3,MMM,Q1 Earnings Roundup: 3M (NYSE:MMM) And The Res...,Quarterly earnings results are a good time to ...,2025-05-22,StockStory,https://finance.yahoo.com/news/q1-earnings-rou...
4,MMM,3 Cash-Producing Stocks with Questionable Fund...,While strong cash flow is a key indicator of s...,2025-05-19,StockStory,https://finance.yahoo.com/news/3-cash-producin...
...,...,...,...,...,...,...
4866,ZTS,2 Dividend Stocks to Buy With $500 and Hold Fo...,Zoetis is a leading animal health company with...,2025-05-23,Motley Fool,https://www.fool.com/investing/2025/05/23/2-di...
4867,ZTS,Zoetis (NYSE:ZTS) Declares US$0.50 Dividend Pe...,Zoetis (NYSE:ZTS) recently affirmed a dividend...,2025-05-22,Simply Wall St.,https://finance.yahoo.com/news/zoetis-nyse-zts...
4868,ZTS,Jim Cramer on Zoetis (ZTS): “It Does Seem to B...,We recently published a list of Jim Cramer Tal...,2025-05-21,Insider Monkey,https://finance.yahoo.com/news/jim-cramer-zoet...
4869,ZTS,Zoetis (ZTS) Upgraded to Buy: Here's Why,Zoetis (ZTS) might move higher on growing opti...,2025-05-21,Zacks,https://finance.yahoo.com/news/zoetis-zts-upgr...


In [ ]:
df_news['EMBEDDED_TEXT'] = df_news['TITLE'] + ' : ' + df_news['SUMMARY']

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Implement FAISS vector store
We:
- Use a pre-trained sentence transformer (all-MiniLM-L6-v2) to embed documents.
- Normalize vectors to use cosine similarity.
- Create a FAISS index and implement a basic search function.

This will allow us to retrieve relevant news snippets given a natural language question.


In [ ]:
# Load model and compute embeddings
text_embeddings = model.encode(df_news['EMBEDDED_TEXT'].tolist(), convert_to_numpy=True)

# Normalize embeddings to use cosine similarity (via inner product in FAISS)
text_embeddings = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)

# Prepare metadata
documents = df_news['EMBEDDED_TEXT'].tolist()
metadata = [
    {
        'PUBLICATION_DATE': row['PUBLICATION_DATE'],
        'TICKER': row['TICKER'],
        'PROVIDER': row['PROVIDER']
    }
    for _, row in df_news.iterrows()
]

In [ ]:
embedding_dim = text_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)  # Cosine similarity via inner product
faiss_index.add(text_embeddings)

In [ ]:
class FaissVectorStore:
    def __init__(self, model, index, embeddings, documents, metadata):
        self.model = model
        self.index = index
        self.embeddings = embeddings
        self.documents = documents
        self.metadata = metadata

    def search(self, query, k=5, metadata_filter=None):
        query_embedding = self.model.encode([query])
        query_embedding = query_embedding / np.linalg.norm(query_embedding)

        if metadata_filter:
            filtered_indices = [i for i, meta in enumerate(self.metadata) if metadata_filter(meta)]
            if not filtered_indices:
                return []
            filtered_embeddings = self.embeddings[filtered_indices]
            temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
            temp_index.add(filtered_embeddings)
            D, I = temp_index.search(query_embedding, k)
            indices = [filtered_indices[i] for i in I[0]]
        else:
            D, I = self.index.search(query_embedding, k)
            indices = I[0]
            D = D[0]

        results = []
        for idx, sim in zip(indices, D):
            results.append((self.documents[idx], self.metadata[idx], float(sim)))
        return results

In [ ]:
# Create FAISS-based store
faiss_store = FaissVectorStore(
    model=model,
    index=faiss_index,
    embeddings=text_embeddings,
    documents=documents,
    metadata=metadata
)

### Setup OpenAI Client

👉 **Instructions**:
- Import the `OpenAI` client from the `openai` Python library.
- You will need an **OpenAI API key** to use their models programmatically:
  - Go to [https://platform.openai.com/](https://platform.openai.com/) and sign up or log in.
  - Create an API key from your [API keys dashboard](https://platform.openai.com/account/api-keys).
  - ⚠️ **Keep your API key private** and **do not** share or hardcode it in public notebooks.
- Note that **usage of the OpenAI API is not free**. You will need to:
  - Add a payment method.
  - Monitor your usage to avoid unexpected charges.
  - Optionally set usage limits from your account settings.
- You can refer to the **course’s Study Resources** for a step-by-step guide on creating an OpenAI account and retrieving your API key.

Then:
- Initialize the client with `OpenAI(api_key="YOUR_KEY_HERE")`.
- Send a test request using `.responses.create()` and the `"gpt-4o-mini"` model with a simple prompt:

  ```python
  response = client.responses.create(
      model="gpt-4o-mini",
      input="Write a one-sentence bedtime story about a unicorn."
  )
  print(response.output_text)


In [ ]:
sk_gpt='/content/drive/MyDrive/Colab Notebooks/sk_gpt.py'
exec(open(sk_gpt).read())

In [ ]:
# CODE HERE
# Use as many coding cells as you need
from openai import OpenAI

# Now that 'my_sk' is defined, you can use it to initialize the OpenAI client
from openai import OpenAI

client = OpenAI(api_key=my_sk)

response = client.responses.create(
    model="gpt-4o-mini",
    input="Write a one-sentence bedtime story about a unicorn."
)
print(response.output_text)


As the moonlight danced on the enchanted forest, a gentle unicorn named Luna shared her shimmering rainbow with weary dreamers, filling their hearts with magic and kindness as they drifted into slumber.


## Retrieve Additional Metadata from Yahoo Finance

👉 **Instructions**:
- We will enrich our news dataset by retrieving **company-level metadata** using the `yfinance` library.
- The goal is to map each unique stock ticker (`TICKER`) in the dataset to:
  - `COMPANY_NAME`
  - `SECTOR`
  - `INDUSTRY`

> ℹ️ `yfinance` fetches live data from Yahoo Finance. If you're running this in a cloud environment or during peak hours, expect some tickers to fail or rate limits to apply.

✅ After this step, you will have a new DataFrame (e.g. `df_meta`) with the columns `TICKER`, `COMPANY_NAME`, `SECTOR`, `INDUSTRY` that maps tickers to their company names, sectors, and industries. This metadata will be useful later to add filters and analysis based on sector or industry categories.


In [ ]:
#get df_meta

df_meta=pd.DataFrame(columns=['TICKER', 'COMPANY_NAME', 'SECTOR', 'INDUSTRY'])
df_meta['TICKER']=df_news['TICKER'].unique()
df_meta.head()


,TICKER,COMPANY_NAME,SECTOR,INDUSTRY
0,MMM,NaN,NaN,NaN
1,AOS,NaN,NaN,NaN
2,ABT,NaN,NaN,NaN
3,ABBV,NaN,NaN,NaN
4,ACN,NaN,NaN,NaN


In [ ]:
#get the info for each ticker
import yfinance as yf
import time

for t in df_meta['TICKER']:
  df_meta.loc[df_meta['TICKER']==t,'COMPANY_NAME']=yf.Ticker(t).info['longName']
  df_meta.loc[df_meta['TICKER']==t,'SECTOR']=yf.Ticker(t).info['sector']
  df_meta.loc[df_meta['TICKER']==t,'INDUSTRY']=yf.Ticker(t).info['industry']
  time.sleep(0.5)

df_meta.head()

,TICKER,COMPANY_NAME,SECTOR,INDUSTRY
0,MMM,3M Company,Industrials,Conglomerates
1,AOS,A. O. Smith Corporation,Industrials,Specialty Industrial Machinery
2,ABT,Abbott Laboratories,Healthcare,Medical Devices
3,ABBV,AbbVie Inc.,Healthcare,Drug Manufacturers - General
4,ACN,Accenture plc,Technology,Information Technology Services


## Retrieval-Augmented Generation (RAG): Retrieve Documents and Generate Answers

👉 **Instructions**:

In this part of the assignment, your task is to build a simple Retrieval-Augmented Generation (RAG) pipeline that:

- Takes a user question as input.
- Searches the FAISS vector store to find a set of relevant financial news articles based on semantic similarity.
- Uses the retrieved news articles as context to generate a clear, concise answer to the question by interacting with the OpenAI language model.
- Returns both the generated answer and the underlying news snippets used for context.

### What you need to focus on:

- Implement a retrieval mechanism to query your vector store and obtain the top relevant documents for any question.
- Construct prompts that effectively combine retrieved news content with the user’s question to guide the language model’s response.
- Use the OpenAI API to generate answers grounded in the retrieved context.
- Organize the outputs so that for each question, you have:
  - The generated answer.
  - The collection of news excerpts used to produce that answer.

### What you will be provided:

- Helper functions to display outputs in markdown format.
- Lists of example questions covering topics, companies, and industries to test your implementation.

---

Your solution can take any form or structure you find appropriate, as long as it fulfills these core objectives. This exercise will give you hands-on experience with integrating retrieval and generation for practical applications in finance.


#### Print markdown
You can use the following function to print answers from GPT4o-mini in markdown.

In [ ]:
from IPython.display import Markdown, display

def print_markdown(text):
    display(Markdown(text))

#### Predefined questions

In [ ]:
questions_topic = [
"What are the major concerns expressed in financial news about inflation?",
"How is investor sentiment described in recent financial headlines?",
"What role is artificial intelligence playing in recent finance-related news stories?"
]

questions_company = [
"How is Microsoft being portrayed in news stories about artificial intelligence?",
"What financial news headlines connect Amazon with automation or logistics?"
]

questions_industry = [
"What are the main themes emerging in financial news about the semiconductor industry?",
"What trends are being reported in the retail industry?",
"What risks or challenges are discussed in recent news about the energy industry?"
]

In [ ]:
# CODE HERE
# Use as many coding cells as you need
def questions(qs):
  #call FAISS VECTORE STORE
  search=faiss_store.search(qs,k=K)
  #give the context to GPT
  context= "\n\n" .join(s[0] for s in search)
  #create the prompt
  prompt= f""" Use the context to answer the following question
  context: {context}
  question: {qs}
  answer: """
  #call GPT
  response = client.responses.create(
      model="gpt-4o-mini",
      input=prompt
  )
  return response

In [ ]:
for q in questions_topic:
  print(q)
  GPT_Answer=questions(q)
  #print(GPT_Answer)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

What are the major concerns expressed in financial news about inflation?


The major concerns expressed in financial news about inflation include:

1. **Persistent Inflation Risks**: The Federal Reserve has shown mounting concern over ongoing inflation in the U.S., indicating fears of economic slowdown.

2. **Food Inflation Impact**: Rising food prices are dampening hopes for interest rate cuts, suggesting that inflationary pressures are particularly affecting basic consumer goods.

3. **Macroeconomic Uncertainty**: There is a general uncertainty regarding the macroeconomic outlook, which is negatively impacting earnings expectations and fostering a cautious investment environment.

4. **Tariff Effects**: The imposition of new tariffs is contributing to increased costs for consumers, exacerbating inflationary trends.

5. **Fiscal Health Concerns**: Increased U.S. national debt and interest payments raise alarms about fiscal sustainability, which may lead to further inflationary pressures.

6. **Investor Flight to Hard Assets**: Investors are turning to hard assets like gold amid inflation fears, indicating a lack of confidence in traditional currency stability.

Overall, these factors contribute to a grim economic outlook, with a strong focus on how inflation is impacting various sectors and influencing financial decisions.


---------------------------------------------------------------------------------------------------------------------------------------

How is investor sentiment described in recent financial headlines?


Investor sentiment in recent financial headlines is characterized by a mix of optimism and caution. Many articles highlight Wall Street's bullish outlook on certain stocks, often accompanied by ambitious price targets that suggest significant upside potential. However, there is also a prevailing skepticism due to analysts' tendencies to avoid issuing bearish ratings, which can lead to overly optimistic forecasts. This cautious sentiment is underscored by reports of downbeat forecasts for some stocks, indicating serious concerns about their fundamentals. Overall, while there are stocks delivering strong performances and attracting positive attention, there are also notable warnings about the potential risks and challenges that investors should be mindful of.


---------------------------------------------------------------------------------------------------------------------------------------

What role is artificial intelligence playing in recent finance-related news stories?


Artificial intelligence (AI) is playing a significant role in recent finance-related news stories by driving innovation, enhancing productivity, and shaping investment strategies. Key aspects include:

1. **Lending and Credit Risk**: Companies like Jack Henry are integrating AI-driven lending technology to streamline processes and reduce human error. Upstart, another AI-focused firm, uses AI to help lenders assess credit risk effectively.

2. **Investment Trends**: Billionaire hedge fund managers are heavily investing in AI stocks, indicating confidence in the sector's potential to generate substantial returns. Stocks like Palantir and Upstart are highlighted for their growth prospects tied to AI applications.

3. **Market Performance**: AI technologies are impacting stock performance, with companies like Palantir and Snowflake showing strong gains, even amid broader market volatility. AI is seen as a major growth driver in the tech sector.

4. **Earnings Insights**: Companies like Salesforce are scrutinized for their heavy focus on AI amidst declining performance in other areas, suggesting a need for balance in adopting these technologies.

5. **Strategic Adaptation**: Businesses like News Corp are shifting towards digital and AI-driven strategies to compete effectively in changing markets, emphasizing the importance of AI in modern operational frameworks.

Overall, AI is not only transforming how finance-related companies operate but also influencing investment decisions and market dynamics.


---------------------------------------------------------------------------------------------------------------------------------------



In [ ]:
for q in questions_industry:
  print(q)
  GPT_Answer=questions(q)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

What are the main themes emerging in financial news about the semiconductor industry?


The main themes emerging in financial news about the semiconductor industry include:

1. **International Revenue Trends**: There's a focus on assessing how companies like ON Semiconductor Corp. are performing internationally and how these trends impact forecasts and stock performance.

2. **Stock Performance Amid Earnings Fluctuations**: Despite some companies reporting soft or negative earnings, such as ON Semiconductor's net loss, their stock prices have surged, indicating investor confidence driven by factors like share buyback programs.

3. **Investor Attention and Optimism**: Stocks within the semiconductor sector are drawing significant investor interest, with major investors like Billionaire Glenn Russell Dubin recognizing companies like ON Semiconductor as having substantial upside potential.

4. **Performance Comparison**: The semiconductor companies are often compared based on recent financial performances, with some seeing sharp price increases despite broader market declines, suggesting resilience in certain segments of the industry.

5. **Challenges and Opportunities**: While some companies face declining demand in sectors like electric vehicles, they continue to show growth in areas such as silicon carbide (SiC) and AI data centers.

6. **Market Trends**: The semiconductor industry's recovery is seen in light of strong demand from sectors like AI and the overall economic environment, which influences stock forecasts and investment strategies.


---------------------------------------------------------------------------------------------------------------------------------------

What trends are being reported in the retail industry?


The retail industry is experiencing several concerning trends:

1. **Volatility in Demand**: Consumer spending has been inconsistent, leading to a drop in retail stocks by 13.7% over the past six months, significantly worse than the S&P 500's 5.5% loss.

2. **Supply Chain Adjustments**: Retailers are adapting their supply chains in response to trade policies and tariffs, with many price increases already implemented.

3. **Inventory Challenges**: There is a potential for inventory overflow due to increased imports ahead of a tariff pause, which could trigger deeper discounts and margin pressure for retailers.

4. **Economic Cycles Impact**: The performance of consumer discretionary businesses is closely tied to economic conditions, and the industry has seen a downturn of 12.3% over the last six months, also worse than the S&P 500.

5. **High Operating Costs**: Restaurants, being part of the retail sector, face high inventory and labor costs, leading to thin margins, making them particularly vulnerable if demand decreases. The restaurant industry has tumbled by 13% in the same period.

6. **Consumer Staples Struggles**: Consumer staples, usually seen as safer investments, have also struggled, facing a 13.9% decline, reflecting broader industry challenges.

Overall, retailers are navigating a complex landscape of fluctuating consumer demand, changing supply chains, and economic pressures, leading to a pessimistic outlook in the sector.


---------------------------------------------------------------------------------------------------------------------------------------

What risks or challenges are discussed in recent news about the energy industry?


Recent news highlights several significant risks and challenges facing the energy industry:

1. **Legislative Risks**: A bill in Congress is advancing that could repeal crucial subsidies for the renewable energy sector, threatening the profitability of renewable projects at a time when companies are increasing production.

2. **Price Volatility**: Oil prices are sliding, leading to tougher conditions for oilfield service companies like SLB, HAL, and BKR. This drop in prices could affect their profitability and operations.

3. **Tariffs and Trade Conflicts**: Rising tariffs and impacts from ongoing trade wars, such as those initiated during the Trump administration, are creating an uncertain environment for U.S. industrial and energy businesses.

4. **Economic Downturn Predictions**: The industrial sector, which is integral to energy production and infrastructure, is experiencing a downturn, with stock values dropping significantly in anticipation of a prolonged economic slump.

5. **Environmental Risks**: Companies like Edison International face lawsuits alleging false statements regarding wildfire risk management, which could impact their reputation and financial stability.

6. **Valuation Concerns**: Many mid- and small-cap oil and gas stocks are trading below their book values, indicating potential undervaluation but also reflecting underlying market challenges.

These factors combine to create a complex landscape for investors and companies in the energy sector.


---------------------------------------------------------------------------------------------------------------------------------------



In [ ]:
for q in questions_company:
  print(q)
  GPT_Answer=questions(q)
  #print(GPT_Answer)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

How is Microsoft being portrayed in news stories about artificial intelligence?


Microsoft is portrayed as a major player in the artificial intelligence (AI) landscape, leading innovations and advancements in the market. The articles highlight the company's strategic partnerships and collaborations with other tech giants, which bolster its position in AI. Additionally, Microsoft's efforts to develop AI technologies and integrate them into various applications suggest that it is actively expanding its role in the growing AI sector. Overall, Microsoft is depicted as a significant contributor to and beneficiary of the AI revolution, poised for continued growth and impact in this domain.


---------------------------------------------------------------------------------------------------------------------------------------

What financial news headlines connect Amazon with automation or logistics?


The financial news headlines that connect Amazon with automation or logistics are:

1. **UPS Sells Ware2Go To Peter Thiel-Backed Stord** - This headline mentions Stord acquiring UPS's Ware2Go as part of its strategy to compete with Amazon in e-commerce, indicating a focus on logistics.

2. **Nvidia can't be stopped, Apple falls behind, and the AI data center race** - Although this headline focuses on AI, it implies that Amazon Web Services is reconsidering some leases, hinting at potential adjustments in their logistics and automation strategies amidst market changes.

These headlines showcase Amazon's involvement in logistics and highlight its competitive landscape in automation.


---------------------------------------------------------------------------------------------------------------------------------------



## Analysis & Questions - Section 1

### Analysis and Reflection on Retrieval and Generation Results
After running the RAG pipeline and obtaining answers along with their supporting news excerpts, take some time to carefully review both the generated responses and the retrieved contexts.

- **For each question, read the answer and then the corresponding news snippets used as context.**

- Reflect on the following points and document your observations:
1. **Relevance**
2. **Completeness**  
3. **Bias or Noise**
4. **Consistency**  
5. **Improvement Ideas**   

and answer the questions below:

#### **Question 1.** How well do the retrieved news snippets support the generated answer? Are the key facts or themes in the answer clearly grounded in the context?

The key themes are clearly related to the question context, but it si like if I'm reading a very specilized financial newspaper. I get the general context related to the news, but nothing very specific.




#### **Question 2.** Does the answer fully address the question, or does it leave important aspects out? Consider if the retrieved context provided enough information to generate a thorough response.

It gives a very general contexto of the question, but it leaves important factors out of the answer. When it answers the question:



> What financial news headlines connect Amazon with automation or logistics? The financial news headlines that connect Amazon with automation or logistics include:
> UPS Sells Ware2Go to Peter Thiel-Backed Stord - This article discusses how Stord is acquiring Ware2Go from UPS to increase its fulfillment space and compete with Amazon, highlighting the logistics aspect of Amazon's operations.
> Every aspect of Amazon is leveraging artificial intelligence - This commentary from Matt Garman, CEO of Amazon Web Services, indicates that automation and AI are integral to Amazon's operations.
> These headlines underscore Amazon's involvement in logistics and automation, particularly in the context of competition and technological advancements.

It just answers that Amazon has a major competitor for logistics, which is a 3PL, and then it goes with AWS strategy. But it doesn´t gives more specific data or justifies why this is important.





#### **Question 3.** Are there any irrelevant or misleading snippets retrieved that may have influenced the answer? How might this affect the quality of the output?

Yes, but I don´t think that is the question, I think is the lack of context that we are sending to the GPT, and that is what affects the output. If we give more context, the quality of the output is mucho better. I know this, cause I made the same excercise question in the next steps (aditionally I added my questions), and the utput with the enriched metadata was much better.

#### **Question 4.**  Do the news snippets show consistent information, or are there conflicting viewpoints? How does the LLM handle potential contradictions in the context?

For the question:

> How is investor sentiment described in recent financial headlines?

> Investor sentiment in recent financial headlines is characterized by a mix of **bullish optimism and cautious skepticism**. Many articles highlight that Wall Street remains overwhelmingly bullish on various stocks, with ambitious price targets suggesting significant upside potential. However, *there's a recurring theme of skepticism regarding the credibility of these forecasts*, as analysts may be influenced by institutional pressures or fear of jeopardizing other business opportunities like M&A advisory.

>Additionally, **there's attention on unpopular or bearish forecasts**, indicating that when Wall Street turns negative on a stock, it should be taken seriously, as such ratings are uncommon due to the industry’s tendency to maintain positive corporate relationships. Overall, investors are advised to navigate this landscape with caution, weighing both the exciting developments and underlying fundamentals of the stocks being discussed.

It has two main ideas that have a conflicting point of view. Cause for one side investors are bullish, which means that stock prices will rise and it's time for buying, but also, in the second paragraph, it says that there are someones taht are expecting that stock prices will go down (bearish). Also, there is a third idea, that it skepticism, whis means that investors doesn´t belive that the bullish scenario will come true.

I fell like the LLM is just trying to say the same thing that a person that is not a financial will say, in order to handle both ideas that contradict each other.


#### **Question 5.**  Based on your observations, suggest ways the retrieval or generation process could be improved (e.g., better filtering, adjusting `k`, refining prompt design).

I think that it should be with more context and making better prompots, cause if you give an incomplete context to the AI, the answer will be very generic or misleaded to your purpose.

If we only focused on the K or other parametrs, we could imporove our answer, but not the quality of the answer. I mean, with the hyperparameters adjustments, qwe could only improve the quality of the answer +/-1% or something near to that, if we enrich the context, we have a better performance.

## 🧠 Retrieval-Augmented Generation (RAG) v2: Adding Financial Metadata to Improve Generation

👉 **Instructions**:

In this part of the assignment, you’ll enhance your Retrieval-Augmented Generation (RAG) pipeline by incorporating *financial metadata* to provide more contextually rich answers.

Your goal is to evaluate whether metadata such as **company name**, **sector**, and **industry** helps the LLM generate **more accurate and grounded answers** to financial questions.

---

### ✅ What your updated pipeline should do:

- Retrieve relevant financial news articles using semantic similarity with FAISS.
- Enrich each retrieved document with financial metadata:
  - Ticker symbol
  - Full company name
  - Sector (e.g., Technology, Energy)
  - Industry (e.g., Semiconductors, Retail)
- Construct prompts that include both:
  - Retrieved news text
  - Associated metadata
- Send the prompt to the OpenAI model to generate an informed response.
- Return:
  - The final answer
  - The exact set of contextual documents used to produce that answer

---

### 🧪 Evaluation and Comparison:

You will test your improved RAG pipeline on the same three types of questions provided earlier:
- **Topic-focused** (e.g., inflation, interest rates)
- **Company-focused** (e.g., questions about Tesla, Nvidia)
- **Industry-focused** (e.g., semiconductors, utilities)


In [ ]:
# CODE HERE
# Use as many coding cells as you need
def questions_metadata(qs, metdata):
    search = faiss_store.search(qs, k=K)
    #as in the last exercise, I create a list of dictionaries to enrich
    enriched_context = []
    for doc in search:
        met=df_meta[df_meta['TICKER']==doc[1]['TICKER']]
        ticker=met['TICKER'].values[0] if not met.empty else None
        if ticker and ticker in df_meta["TICKER"].values:
            meta_row = df_meta[df_meta["TICKER"] == ticker].iloc[0]
            enriched_context.append(
                f"{doc[0]}\n"
                f"[TICKER: {meta_row['TICKER']}, COMPANY: {meta_row['COMPANY_NAME']}, "
                f"SECTOR: {meta_row['SECTOR']}, INDUSTRY: {meta_row['INDUSTRY']}]"
            )
        else:
            enriched_context.append(doc[0])


    prompt = f"""Use the context and metadata to answer the following question.
            Context:{enriched_context}
            Question:{qs}
            Answer:"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )
    return response

In [ ]:
for q in questions_topic:
  print(q)
  GPT_Answer=questions_metadata(q, df_meta)
  #print(GPT_Answer)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

What are the major concerns expressed in financial news about inflation?


The major concerns expressed in financial news about inflation include:

1. **Persistent Inflation Risks**: The Federal Reserve's minutes indicated growing worries about sustained inflation levels in the U.S., coupled with the potential for an economic slowdown.

2. **Food Inflation**: Rising food prices have dampened expectations for a rate cut, with new tariffs contributing to higher grocery bills, impacting consumer spending.

3. **Uncertain Economic Outlook**: Analysts are concerned that the overall macroeconomic environment remains uncertain, causing a drag on earnings expectations and leading to potential downgrades in stock performance.

4. **Geopolitical Tensions**: Increasing geopolitical issues and skepticism about fiscal sustainability are driving investors toward hard assets, reflecting worries about currency devaluation and long-term inflation.

5. **Credit Rating Downgrades**: Recent downgrades of the U.S. sovereign credit rating have heightened fears about the nation's fiscal health and long-term economic stability. 

Overall, these factors are creating a cautious atmosphere among investors and analysts regarding future economic conditions.


---------------------------------------------------------------------------------------------------------------------------------------

How is investor sentiment described in recent financial headlines?


Investor sentiment in recent financial headlines is characterized by a mix of optimism and caution. Many articles highlight bullish perspectives from Wall Street, suggesting attractive upside potential for several stocks, often backed by ambitious price targets. However, there's also a recurring theme of skepticism, as analysts are noted to face pressures that may lead to overly optimistic forecasts. Additionally, there are several mentions of bearish sentiments, with some stocks receiving downbeat ratings due to various challenges, which analysts typically issue rarely to maintain relationships for future business opportunities. Overall, while there is excitement about certain stocks, there's a strong emphasis on the need for caution and critical evaluation of the forecasts.


---------------------------------------------------------------------------------------------------------------------------------------

What role is artificial intelligence playing in recent finance-related news stories?


Artificial intelligence (AI) is playing a significant role in recent finance-related news stories by enhancing productivity, automating decision-making, and improving credit risk assessment. Companies like Upstart are leveraging AI to quantify credit risk for lenders, while Jack Henry integrates AI-driven lending technologies to boost efficiency and reduce human error.

Furthermore, finance leaders are increasingly focusing on practical AI adoption to revamp operations, as shown in discussions at the Gartner CFO conference. AI is also seen as a key factor in driving investment decisions, where stocks associated with AI applications are being closely monitored for their potential upside, as evidenced by billionaire hedge funds investing in AI stocks like Palantir and Upstart. 

Overall, AI is transforming the financial landscape by reducing operational inefficiencies and providing strategic benefits that are attractive to investors.


---------------------------------------------------------------------------------------------------------------------------------------



In [ ]:
for q in questions_industry:
  print(q)
  GPT_Answer=questions_metadata(q, df_meta)
  #print(GPT_Answer)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

What are the main themes emerging in financial news about the semiconductor industry?


The main themes emerging in financial news about the semiconductor industry include:

1. **Revenue Trends**: There is a focus on the international revenue trends of semiconductor companies, as seen with ON Semiconductor Corp. This highlights the importance of global markets in shaping the financial outlook and forecasts for these companies.

2. **Investor Attention**: Companies like ON Semiconductor are attracting significant attention from investors, indicating a renewed interest in semiconductor stocks amidst various market dynamics.

3. **Earnings Performance**: Despite some companies, like ON Semiconductor, reporting soft earnings or net losses, there is still optimism among shareholders, suggesting that market perceptions may not fully align with immediate financial performance.

4. **Market Response and Stock Movements**: Notably, substantial stock price movements occur in reaction to buyback programs and other corporate actions. For instance, ON Semiconductor experienced a significant stock price increase despite reporting losses, indicating investor confidence in its strategic decisions.

5. **Comparative Analysis with Industry Peers**: Articles often compare different semiconductor firms, discussing their performance relative to market trends and each other, thus providing a clearer picture of the competitive landscape.

6. **Impact of Macroeconomic Conditions**: The industry is also facing challenges due to macroeconomic conditions affecting demand, particularly in electric vehicles (EVs). Companies are adapting strategies to maintain growth in areas like silicon carbide (SiC) and AI data centers, reflecting a shift in focus within the industry.

These themes collectively suggest that the semiconductor sector is navigating a complex environment where investor sentiment, revenue trends, and strategic business decisions play critical roles in shaping its future.


---------------------------------------------------------------------------------------------------------------------------------------

What trends are being reported in the retail industry?


The retail industry is experiencing significant challenges, with several trends reported:

1. **Adapting Business Models**: Retailers are adjusting their strategies in response to technological changes affecting consumer shopping habits.

2. **Volatility in Demand**: There is noted volatility in demand, which is impacting consumer spending and contributing to uncertainty in the market.

3. **Declining Stock Performance**: Over the past six months, retail stocks have dropped by 13.7%, significantly worse than the S&P 500’s 5.5% loss.

4. **Supply Chain Adjustments**: Retailers are modifying their supply chains to navigate the impacts of tariffs and trade policies, with many price increases already affecting consumers.

5. **Inventory Challenges**: Concerns are rising over potential inventory overflows due to a surge in imports ahead of a tariff pause, leading to fears of deeper discounts and squeezed margins.

6. **Economic Cycle Sensitivity**: The performance of consumer discretionary businesses is closely tied to economic cycles, with recent trends suggesting declining demand.

7. **High Costs**: A focus on thin margins due to high inventory and labor costs is prevalent, making restaurants and other retail sectors particularly vulnerable if demand decreases further.

8. **Performance of Different Sectors**: Overall, the consumer staples sector has faltered, with a drop of 13.9% over six months, indicating broader challenges across various retail categories.

These trends reflect a complex landscape for retailers as they navigate economic pressures and shifting consumer behaviors.


---------------------------------------------------------------------------------------------------------------------------------------

What risks or challenges are discussed in recent news about the energy industry?


Recent news highlights several risks and challenges in the energy industry:

1. **Renewable Energy Subsidy Reductions**: A bill passed by the U.S. House of Representatives could repeal crucial subsidies for the renewable energy sector, making projects uneconomical and leading to a crash in renewable energy stocks. This presents a significant risk to companies like NextEra Energy and Enphase Energy.

2. **Oilfield Service Sector Challenges**: Companies in the oilfield service sector, such as Halliburton and Baker Hughes, are facing difficulties as oil prices decline, tariffs increase, and drilling budgets shrink.

3. **Economic Uncertainty in Industrials**: The industrial sector, which includes energy-related industries, is facing a downturn, with significant drops in stock performance and forecasts indicating a prolonged economic slowdown.

4. **Political and Regulatory Risks**: The potential effects of Trump’s policies on energy projects, specifically offshore wind efforts by Dominion Energy, create uncertainty for future developments in renewable energy.

5. **Wildfire Liability and Litigation**: Utilities like Xcel Energy and Edison International are dealing with lawsuits and litigation over wildfire risks, adding financial and operational pressures.

6. **Market Challenges**: The energy sector overall is reported to be undervalued, with certain stocks trading below book value, suggesting potential investment challenges amidst a broader economic context.

These risks indicate a tumultuous environment for energy companies, affecting both renewable and traditional energy sectors.


---------------------------------------------------------------------------------------------------------------------------------------



In [ ]:
for q in questions_company:
  print(q)
  GPT_Answer=questions_metadata(q, df_meta)
  #print(GPT_Answer)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

How is Microsoft being portrayed in news stories about artificial intelligence?


Microsoft is portrayed positively in news stories about artificial intelligence. The company is highlighted as a key player in the AI revolution, collaborating with other tech giants like Amazon and benefiting from increasing demand for AI-driven solutions. Reports suggest that Microsoft is involved in significant innovations in AI, particularly within the software infrastructure sector, and demonstrates resilience as it regains momentum amid market competition. Overall, Microsoft's strategic partnerships and advancements in AI contribute to a favorable outlook in the context of the AI market growth.


---------------------------------------------------------------------------------------------------------------------------------------

What financial news headlines connect Amazon with automation or logistics?


The financial news headlines that connect Amazon with automation or logistics are:

1. **"UPS Sells Ware2Go To Peter Thiel-Backed Stord As Startup Gains 2.5M Square Feet To Compete With Amazon"** - This headline discusses how Stord, a logistics tech startup, is expanding its fulfillment space to challenge Amazon in the e-commerce sector.

2. **"Amazon's AI Roadmap With AWS CEO Garman"** - Although this focuses on AI, the integration of artificial intelligence often relates to automation in logistics, as Amazon leverages AI across its various operations.

These headlines illustrate Amazon's engagement in the logistics and automation landscape.


---------------------------------------------------------------------------------------------------------------------------------------



In [ ]:
df_meta['SECTOR'].unique()

array(['Industrials', 'Healthcare', 'Technology', 'Utilities',
       'Financial Services', 'Basic Materials', 'Consumer Cyclical',
       'Real Estate', 'Communication Services', 'Consumer Defensive',
       'Energy'], dtype=object)

In [ ]:
questions_topic = [
"What effect has the unemployment rate for FOREX market?",
"Is AI playing an important role in trading?",
"How does macroeconomic indicators affect the stock market ?"
]

questions_company = [
"Is Intel going to integrate AI to its processors?",
"Does Salesforce announce any updates related to AI to the software?",
"Which are the pain points that FISERV has?"
]

questions_industry = [
"How fast is Medical Devices industry adopting AI?",
"How does AI is changing the Information Technology Services?",
"Which are the major achievments for hydrogen in the energy industry?"
]

In [ ]:
for q in questions_topic:
  print(q)
  GPT_Answer=questions_metadata(q, df_meta)
  #print(GPT_Answer)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

What effect has the unemployment rate for FOREX market?


The unemployment rate significantly impacts the FOREX market as it is a critical indicator of economic health. A strong jobs market often leads to higher consumer spending, which can bolster economic growth. This typically strengthens the currency, as investors are more likely to buy a currency from a country with low unemployment. Conversely, high unemployment can lead to a weaker currency due to concerns over economic downturns and reduced consumer confidence. Consequently, FOREX traders closely monitor unemployment data to make informed trading decisions, as changes in the employment landscape can influence interest rates set by central banks and cause fluctuations in currency values.


---------------------------------------------------------------------------------------------------------------------------------------

Is AI playing an important role in trading?


Yes, AI is playing an important role in trading. The context indicates that various companies, particularly those in the technology and finance sectors, are leveraging AI for improved decision-making, integration of AI capabilities into operations, and optimizing costs. Investors are increasingly looking at AI stocks for long-term growth potential, with mentions of performance gains for companies like Palantir and the rising tide of AI-related stocks. Additionally, AI's ability to drive productivity and reduce human error is highlighted, suggesting its significant impact on trading strategies and market dynamics.


---------------------------------------------------------------------------------------------------------------------------------------

How does macroeconomic indicators affect the stock market ?


Macroeconomic indicators significantly influence the stock market by impacting investor sentiment and corporate earnings expectations. When macroeconomic conditions are uncertain or negative, like rising interest rates or economic slowdowns, companies typically face decreased demand for their products and services. This can lead to analysts lowering their earnings estimates, contributing to bearish market trends.

For example, the industrial sector has seen declines in stock performance due to concerns about capital spending influenced by macroeconomic factors. Similarly, consumer discretionary stocks may suffer during economic downturns, as consumer confidence decreases and spending slows. Overall, negative macroeconomic indicators can prompt investors to be cautious, leading to stock price drops across various sectors.

Conversely, positive macroeconomic indicators, such as robust GDP growth or low unemployment, can enhance investor confidence, driving stock prices higher as earnings projections improve. Thus, the relationship between macroeconomic indicators and the stock market is critical, as these economic factors can dictate market trends and investment strategies.


---------------------------------------------------------------------------------------------------------------------------------------



In [ ]:
for q in questions_company:
  print(q)
  GPT_Answer=questions_metadata(q, df_meta)
  #print(GPT_Answer)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

Is Intel going to integrate AI to its processors?


Yes, Intel is working on developing full-stack AI solutions to enable the next wave of AI-powered computing. This indicates that Intel is indeed planning to integrate AI capabilities into its processors.


---------------------------------------------------------------------------------------------------------------------------------------

Does Salesforce announce any updates related to AI to the software?


The provided context does not mention any updates from Salesforce related to AI in its software. It focuses on various companies and developments in the AI market, but Salesforce is not mentioned.


---------------------------------------------------------------------------------------------------------------------------------------

Which are the pain points that FISERV has?


Fiserv is currently facing several challenges, or "pain points," including:

1. **Leadership Transition**: The recent appointment of Michael P. Lyons as CEO, replacing Frank J. Bisignano, has led to uncertainties surrounding the company's direction and leadership stability.

2. **Stock Price Decline**: Fiserv's shares have decreased significantly, around 12% over the past week, contrasting with broader market gains. This decline could indicate market dissatisfaction or concerns about the company's future performance.

3. **Growth Projections**: The CFO has indicated that the growth of its Clover point-of-sale system is expected to remain flat, which raises concerns about the company's ability to generate increased revenue.

4. **Competitive Pressures**: The company faces competitive pressures in its sector, particularly regarding recurring revenues and the effectiveness of its strategic initiatives.

5. **Lack of Dividends**: There is a noted absence of dividends, which can affect investor sentiment and attractiveness for income-focused investors.

These factors combined may contribute to a challenging environment for Fiserv as it navigates through its current strategic and operational landscape.


---------------------------------------------------------------------------------------------------------------------------------------



In [ ]:
for q in questions_industry:
  print(q)
  GPT_Answer=questions_metadata(q, df_meta)
  #print(GPT_Answer)
  print_markdown(GPT_Answer.output_text)
  print('\n---------------------------------------------------------------------------------------------------------------------------------------\n')

How fast is Medical Devices industry adopting AI?


The adoption of AI in the Medical Devices industry is indicated by companies like Intuitive Surgical, which is competing in the robotic surgery space, and Danaher Corporation, which is developing AI-assisted diagnostic tools. Both companies are integrating AI to enhance their products' capabilities, pointing to a growing trend. However, specific metrics or timelines are not detailed in the context provided, suggesting that while there is increasing interest and investment in AI technologies, the exact speed of adoption may vary across different sectors within the industry. Overall, the Medical Devices field appears to be gradually embracing AI to improve efficiency and outcomes, but comprehensive data on the pace of adoption is lacking in the excerpts.


---------------------------------------------------------------------------------------------------------------------------------------

How does AI is changing the Information Technology Services?


AI is significantly transforming the Information Technology Services sector in several key ways:

1. **Efficiency Improvements**: Companies are leveraging AI to enhance productivity and streamline operations. For instance, Intuit is using AI to improve taxpayer experiences, reducing the time customers spend on returns.

2. **Competitive Edge**: Firms are adopting AI to gain a competitive advantage. Accenture and IBM are adapting their consulting services to align with AI and cloud innovations, which are central to digital transformation strategies for businesses.

3. **Innovative Partnerships**: Companies are partnering with AI technology providers to enhance capabilities. For example, Cognizant Technology Solutions expanded its AI services through partnerships, improving its service offerings and cloud transformation.

4. **Market Growth**: The AI market is projected to grow substantially, driving demand for IT services that facilitate AI integration and usage. This is represented in the expansion plans of major technology firms like Amazon and Salesforce, which are heavily investing in AI technologies.

5. **Challenges and Adaptations**: While there are opportunities, the rapid rise of AI also presents challenges. Companies face increased competition from AI-driven startups, prompting established firms in the IT services sector to innovate and adapt to maintain their market positions.

Overall, AI is becoming a foundational component in enhancing service delivery, fostering innovation, and reshaping competitive dynamics within the Information Technology Services industry.


---------------------------------------------------------------------------------------------------------------------------------------

Which are the major achievments for hydrogen in the energy industry?


The provided context does not specifically mention hydrogen or its achievements in the energy industry. Therefore, I cannot list any major achievements for hydrogen based on the information given. If you would like, I can provide general information about hydrogen's role and achievements in the energy sector outside of the context you provided.


---------------------------------------------------------------------------------------------------------------------------------------



## Analysis & Questions - Section 2

### Instructions: Evaluate Answers With and Without Metadata

For each question, compare the two answers provided:
- One generated **without** metadata
- One generated **with** metadata

---

### Steps:

1. Use the following evaluation criteria:
   - Clarity
   - Detail & Depth
   - Use of Context
   - Accuracy & Grounding
   - Relevance
   - Narrrative Flow

2. For each criterion, write brief notes comparing how the answer **without metadata** performs versus the answer **with metadata**.

3. Summarize your evaluation in a markdown table with the following columns:

| Criteria       | WITHOUT METADATA            | WITH METADATA             |
|----------------|----------------------------|--------------------------|
| Clarity        | Clear but in a general context | It gives a more robust context, which leads to more clearity |
| Detail & Depth         | Generic & cover general context     | More specific and concrete|
| Use of Context        | General context, public knowladge info | More specific and robust |
| Accuracy & Grounding       | Low & general | Specific & considers more variables |
| Relevance      | For a general context | Improves and consider different factors |
| Narrative Flow      | Reading a journalist that isn't specialized in finance | Reading a finance journalist |

---

**Note:** Keep comments short and clear for easy comparison.



YOUR WRITTEN RESPONSE HERE